# SAR Docking Analysis — 구조-활성 관계 자동 분석리간드 시리즈를 자동으로 도킹하고, SAR (Structure-Activity Relationship) 테이블을 생성합니다.## 워크플로우1. **리간드 시리즈 정의** — Core scaffold + R-group 치환체2. **3D 구조 생성** — SMILES → 3D 좌표 (RDKit)3. **자동 도킹** — smina로 전체 시리즈 일괄 도킹4. **SAR Table 생성** — Score, LE, MW, cLogP 등 자동 계산5. **결과 해석** — Activity Cliff, R-group 효과 분석6. **내보내기** — SDF (DataWarrior), CSV, PyMOL 스크립트## 결과물- `sar_results.sdf` → DataWarrior에서 열기- `sar_table.csv` → 스프레드시트 분석- `sar_pymol.pml` → PyMOL에서 바로 실행

## 1. 환경 설정

In [ ]:
#@title 1-1. Install Packages {display-mode: "form"}
!pip install -q rdkit meeko vina openbabel-wheel
!pip install -q pdbfixer openmm pymol-open-source
!pip install -q seaborn pandas matplotlib
print('\n=== Packages installed ===')


### 1-2. Install smina & ADFRsuite

도킹 엔진(smina)과 구조 변환 도구(ADFRsuite)의 바이너리를 다운로드합니다.


In [ ]:
#@title 1-2. Install smina & ADFRsuite {display-mode: "form"}
import os, stat, urllib.request, tarfile, shutil, subprocess

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
os.makedirs(BIN_DIR, exist_ok=True)

# smina
smina_path = f'{BIN_DIR}/smina'
if not os.path.exists(smina_path):
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# ADFRsuite
ADFR_DIR = f'{BIN_DIR}/ADFRsuite'
if not os.path.exists(ADFR_DIR):
    print('Downloading ADFRsuite...')
    urllib.request.urlretrieve('https://ccsb.scripps.edu/adfr/download/1038/', '/tmp/adfr.tar.gz')
    with tarfile.open('/tmp/adfr.tar.gz') as tar:
        tar.extractall('/tmp/')
    d = [x for x in os.listdir('/tmp/') if x.startswith('ADFRsuite')][0]
    try:
        os.chdir(f'/tmp/{d}')
        subprocess.run(['bash', 'install.sh', '-d', ADFR_DIR, '-c', '0'], input=b'Y\n', capture_output=True)
    finally:
        os.chdir('/content')
    for tool in ['prepare_receptor', 'prepare_ligand']:
        src = f'{ADFR_DIR}/bin/{tool}'
        dst = f'{BIN_DIR}/{tool}'
        if os.path.exists(src) and not os.path.exists(dst):
            with open(dst, 'w') as f:
                f.write(f'#!/bin/bash\n{src} "$@"\n')
            os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC)

os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('=== Tools ready ===')


### 1-3. Imports

분자 도킹 파이프라인에 필요한 라이브러리를 불러옵니다.


In [ ]:
#@title 1-3. Imports {display-mode: "form"}
import warnings
warnings.filterwarnings('ignore')

try:
    from pymol import cmd
    HAVE_PYMOL = True
except ImportError:
    HAVE_PYMOL = False
    print("pymol not available - using mol_utils fallback")
    import sys; sys.path.insert(0, "/workspace")
    import mol_utils
from vina import Vina
from openbabel import pybel
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools, Descriptors, rdMolAlign
from pdbfixer import PDBFixer
from openmm.app import PDBFile

import os, subprocess
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
WORK_DIR = '/content/sar_docking'
os.makedirs(WORK_DIR, exist_ok=True)

print('Libraries imported.')


## 2. 리간드 시리즈 정의아래 `LIGANDS` 리스트에 SMILES와 R-group 정보를 입력하세요.예시는 Quinazoline scaffold 기반 EGFR 억제제 시리즈입니다.

In [ ]:
#@title 2-1. 타겟 단백질 설정 {display-mode: "form"}
TARGET_PDB = "6nzp"   #@param {type:"string"}
TARGET_CHAIN = "A"     #@param {type:"string"}

print(f'Target: {TARGET_PDB} (chain {TARGET_CHAIN})')


### 2-2. 리간드 시리즈 정의

2-2. 리간드 시리즈 정의


In [ ]:
#@title 2-2. 리간드 시리즈 정의 {display-mode: "form"}

# ============================================================
# 여기에 본인의 리간드 시리즈를 입력하세요!
# 각 화합물에 name, smiles, 그리고 R-group 정보를 기입합니다.
# ============================================================

CORE_NAME = "4-Anilinoquinazoline"

LIGANDS = [
    # --- R1 변화 (para 위치) ---
    {"name": "Cpd_01_H",      "R1": "—H",     "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccccc3)c2c1"},

    {"name": "Cpd_02_F",      "R1": "—F",     "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(F)cc3)c2c1"},

    {"name": "Cpd_03_Cl",     "R1": "—Cl",    "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(Cl)cc3)c2c1"},

    {"name": "Cpd_04_CH3",    "R1": "—CH₃",   "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(C)cc3)c2c1"},

    {"name": "Cpd_05_OCH3",   "R1": "—OCH₃",  "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(OC)cc3)c2c1"},

    {"name": "Cpd_06_CF3",    "R1": "—CF₃",   "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(C(F)(F)F)cc3)c2c1"},

    {"name": "Cpd_07_NH2",    "R1": "—NH₂",   "R2": "—H",    "series": "R1_scan",
     "smiles": "c1ccc2ncnc(Nc3ccc(N)cc3)c2c1"},

    # --- R2 변화 (meta 위치, R1=F 고정) ---
    {"name": "Cpd_08_F_OH",   "R1": "—F",     "R2": "—OH",   "series": "R2_scan",
     "smiles": "c1cc(O)cc(Nc2ncnc3ccccc23)c1F"},

    {"name": "Cpd_09_F_NH2",  "R1": "—F",     "R2": "—NH₂",  "series": "R2_scan",
     "smiles": "c1cc(N)cc(Nc2ncnc3ccccc23)c1F"},

    {"name": "Cpd_10_F_NHCH3","R1": "—F",     "R2": "—NHCH₃","series": "R2_scan",
     "smiles": "c1cc(NC)cc(Nc2ncnc3ccccc23)c1F"},

    # --- 직접 추가 (예시) ---
    # {"name": "My_cpd", "R1": "—Br", "R2": "—H", "series": "custom",
    #  "smiles": "c1ccc2ncnc(Nc3ccc(Br)cc3)c2c1"},
]

# CSV에서 로드하는 경우:
# df_input = pd.read_csv('my_ligands.csv')  # columns: name, smiles, R1, R2, series
# LIGANDS = df_input.to_dict('records')

print(f'Core scaffold: {CORE_NAME}')
print(f'Total ligands: {len(LIGANDS)}')
print(f'Series: {set(l["series"] for l in LIGANDS)}')


### 2-3. 리간드 2D 구조 확인

도킹 결과의 2D 구조를 그리드로 표시합니다.


In [ ]:
#@title 2-3. 리간드 2D 구조 확인 {display-mode: "form"}
mols = []
legends = []
for lig in LIGANDS:
    mol = Chem.MolFromSmiles(lig['smiles'])
    if mol:
        mols.append(mol)
        legends.append(f"{lig['name']}\nR1={lig['R1']} R2={lig['R2']}")

img = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=5, subImgSize=(300, 220))
img


## 3. 타겟 단백질 준비

In [ ]:
#@title 3-1. PDB 구조 준비 (자동) {display-mode: "form"}
os.chdir(WORK_DIR)

# Step 1: Fetch & clean (pymol or mol_utils)
if HAVE_PYMOL:
    cmd.delete('all')
    cmd.fetch(TARGET_PDB, type='pdb', path=WORK_DIR)
    cmd.remove(f'not chain {TARGET_CHAIN}')
    cmd.select('Prot', 'polymer.protein')
    cmd.select('Lig', 'organic')
    clean_pdb = f'{TARGET_PDB}_clean.pdb'
    lig_mol2 = f'{TARGET_PDB}_lig.mol2'
    cmd.save(clean_pdb, 'Prot')
    cmd.save(lig_mol2, 'Lig')
    cmd.delete('all')
else:
    clean_pdb, lig_mol2 = mol_utils.pdb_clean(TARGET_PDB, chain=TARGET_CHAIN, work_dir=WORK_DIR)

# Step 2: Fix protein (add missing atoms, hydrogens)
prot_H = f'{TARGET_PDB}_clean_H.pdb'
fixer = PDBFixer(filename=clean_pdb)
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(True)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
with open(prot_H, 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# Step 3: Docking box from ligand
if HAVE_PYMOL:
    cmd.delete('all')
    cmd.load(prot_H, 'prot')
    cmd.load(lig_mol2, 'lig')
    ([minX,minY,minZ],[maxX,maxY,maxZ]) = cmd.get_extent('lig')
    pad = 7.0
    center = {'x': (maxX+minX)/2, 'y': (maxY+minY)/2, 'z': (maxZ+minZ)/2}
    size = {'x': maxX-minX+2*pad, 'y': maxY-minY+2*pad, 'z': maxZ-minZ+2*pad}
    cmd.delete('all')
else:
    center, size = mol_utils.get_docking_box(lig_mol2, extending=7.0)
    # Rename keys to match expected format
    center = {'x': center['center_x'], 'y': center['center_y'], 'z': center['center_z']}
    size = {'x': size['size_x'], 'y': size['size_y'], 'z': size['size_z']}

# Step 4: Convert to PDBQT
rec_qt = prot_H.replace('.pdb', '.pdbqt')
prep_rec = os.path.join(BIN_DIR, 'prepare_receptor')
if os.path.exists(prep_rec):
    subprocess.run([prep_rec, '-r', prot_H, '-o', rec_qt], capture_output=True)
else:
    # Fallback: openbabel
    obmol = list(pybel.readfile(format='pdb', filename=prot_H))[0]
    out = pybel.Outputfile(filename=rec_qt, format='pdbqt', overwrite=True)
    out.write(obmol)
    out.close()

print(f'Protein: {prot_H}')
print(f'Receptor PDBQT: {rec_qt}')
print(f'Box center: ({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f})')
print(f'Box size:   ({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')

## 4. 리간드 시리즈 자동 도킹

In [ ]:
#@title 4-1. 전체 시리즈 도킹 (~2-5분) {display-mode: "form"}

EXHAUSTIVENESS = 64   #@param {type:"integer"}
N_POSES = 5           #@param {type:"integer"}

results = []

for i, lig in enumerate(LIGANDS):
    name = lig['name']
    smi = lig['smiles']
    print(f'[{i+1}/{len(LIGANDS)}] {name}...', end=' ')

    # SMILES → 3D SDF
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print('INVALID SMILES')
        continue
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp('_Name', name)

    sdf_in = os.path.join(WORK_DIR, f'{name}.sdf')
    writer = Chem.SDWriter(sdf_in)
    writer.write(mol)
    writer.close()

    # SDF → PDBQT
    pdbqt_in = sdf_in.replace('.sdf', '.pdbqt')
    obmol = list(pybel.readfile(format='sdf', filename=sdf_in))[0]
    out = pybel.Outputfile(filename=pdbqt_in, format='pdbqt', overwrite=True)
    out.write(obmol)
    out.close()

    # smina docking
    sdf_out = os.path.join(WORK_DIR, f'{name}_docked.sdf')
    cmd_args = [
        f'{BIN_DIR}/smina',
        '-r', rec_qt, '-l', pdbqt_in, '-o', sdf_out,
        '--center_x', str(center['x']),
        '--center_y', str(center['y']),
        '--center_z', str(center['z']),
        '--size_x', str(size['x']),
        '--size_y', str(size['y']),
        '--size_z', str(size['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS),
        '--num_modes', str(N_POSES),
    ]
    subprocess.run(cmd_args, capture_output=True)

    # Parse result
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        if suppl and suppl[0] is not None:
            best_mol = suppl[0]
            # Get score from SDF property
            score = None
            for prop in best_mol.GetPropsAsDict():
                if 'affinity' in prop.lower() or 'score' in prop.lower() or 'minimized' in prop.lower():
                    try:
                        score = float(best_mol.GetProp(prop))
                    except:
                        pass
            if score is None:
                try:
                    score = float(best_mol.GetProp('minimizedAffinity'))
                except:
                    score = 0.0

            ha = Chem.RemoveHs(best_mol).GetNumHeavyAtoms()
            mw = Descriptors.MolWt(Chem.RemoveHs(best_mol))

            results.append({
                'Name': name,
                'SMILES': smi,
                'R1': lig.get('R1', ''),
                'R2': lig.get('R2', ''),
                'Series': lig.get('series', ''),
                'Score': round(score, 2),
                'MW': round(mw, 1),
                'cLogP': round(Descriptors.MolLogP(Chem.MolFromSmiles(smi)), 2),
                'HBA': Descriptors.NumHAcceptors(Chem.MolFromSmiles(smi)),
                'HBD': Descriptors.NumHDonors(Chem.MolFromSmiles(smi)),
                'RotBonds': Descriptors.NumRotatableBonds(Chem.MolFromSmiles(smi)),
                'TPSA': round(Descriptors.TPSA(Chem.MolFromSmiles(smi)), 1),
                'HeavyAtoms': ha,
                'LE': round(-score / ha, 3) if ha > 0 else 0,
                'BEI': round(-score / (mw / 1000), 1) if mw > 0 else 0,
                'SDF_file': sdf_out,
                'N_poses': len([m for m in suppl if m is not None]),
            })
            print(f'Score={score:.2f} kcal/mol, LE={results[-1]["LE"]:.3f}')
        else:
            print('No valid poses')
    else:
        print('Docking failed')

print(f'\n=== {len(results)}/{len(LIGANDS)} compounds docked ===')


## 5. SAR Table 생성 & 분석

In [ ]:
#@title 5-1. SAR Table {display-mode: "form"}
df = pd.DataFrame(results)
df = df.sort_values('Score')

# 표시용 포맷팅
display_cols = ['Name', 'R1', 'R2', 'Series', 'Score', 'LE', 'BEI', 'MW', 'cLogP', 'HBA', 'HBD', 'TPSA']
styled = df[display_cols].style \
    .background_gradient(subset=['Score'], cmap='RdYlGn') \
    .background_gradient(subset=['LE'], cmap='YlGn') \
    .format({'Score': '{:.2f}', 'LE': '{:.3f}', 'BEI': '{:.1f}', 'MW': '{:.1f}', 'cLogP': '{:.2f}', 'TPSA': '{:.1f}'})
styled


### 5-2. R-group별 스코어 비교

R-group별 스코어 효과를 분석합니다.


In [ ]:
#@title 5-2. R-group별 스코어 비교 {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R1 scan
r1 = df[df['Series'] == 'R1_scan'].sort_values('Score')
if not r1.empty:
    colors = ['#E53935' if s < -8 else '#43A047' if s < -7 else '#FFB300' for s in r1['Score']]
    axes[0].barh(r1['R1'], r1['Score'], color=colors, edgecolor='black', linewidth=0.5)
    axes[0].set_xlabel('Binding Affinity (kcal/mol)')
    axes[0].set_title(f'R1 Scan (R2 fixed)')
    axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5, label='-7 kcal/mol')
    # Score labels
    for idx, row in r1.iterrows():
        axes[0].text(row['Score'] - 0.1, row['R1'], f'{row["Score"]:.1f}',
                     va='center', ha='right', fontsize=9, fontweight='bold', color='white')
    axes[0].legend()

# R2 scan
r2 = df[df['Series'] == 'R2_scan'].sort_values('Score')
if not r2.empty:
    colors = ['#E53935' if s < -8 else '#43A047' if s < -7 else '#FFB300' for s in r2['Score']]
    axes[1].barh(r2['R2'], r2['Score'], color=colors, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Binding Affinity (kcal/mol)')
    axes[1].set_title(f'R2 Scan (R1=F fixed)')
    axes[1].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### 5-3. Score vs Ligand Efficiency

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 5-3. Score vs Ligand Efficiency {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score vs LE
series_colors = {'R1_scan': '#1E88E5', 'R2_scan': '#FF8F00', 'custom': '#8E24AA'}
for series_name in df['Series'].unique():
    sub = df[df['Series'] == series_name]
    axes[0].scatter(sub['Score'], sub['LE'], s=80, label=series_name,
                    color=series_colors.get(series_name, 'gray'), alpha=0.8, edgecolors='black', linewidth=0.5)
    for _, row in sub.iterrows():
        axes[0].annotate(row['R1'], (row['Score'], row['LE']), fontsize=8, ha='center', va='bottom')
axes[0].axhline(y=0.3, color='green', linestyle='--', alpha=0.5, label='LE=0.3 (drug-like)')
axes[0].set_xlabel('Score (kcal/mol)')
axes[0].set_ylabel('Ligand Efficiency (LE)')
axes[0].set_title('Score vs Ligand Efficiency')
axes[0].legend()

# Score vs MW
for series_name in df['Series'].unique():
    sub = df[df['Series'] == series_name]
    axes[1].scatter(sub['Score'], sub['MW'], s=80, label=series_name,
                    color=series_colors.get(series_name, 'gray'), alpha=0.8, edgecolors='black', linewidth=0.5)
axes[1].axhline(y=500, color='red', linestyle='--', alpha=0.5, label='MW=500 (Lipinski)')
axes[1].set_xlabel('Score (kcal/mol)')
axes[1].set_ylabel('Molecular Weight (Da)')
axes[1].set_title('Score vs Molecular Weight')
axes[1].legend()

plt.tight_layout()
plt.show()


### 5-4. Lipinski Rule of Five 체크

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 5-4. Lipinski Rule of Five 체크 {display-mode: "form"}
df['Lipinski_violations'] = 0
df.loc[df['MW'] > 500, 'Lipinski_violations'] += 1
df.loc[df['cLogP'] > 5, 'Lipinski_violations'] += 1
df.loc[df['HBD'] > 5, 'Lipinski_violations'] += 1
df.loc[df['HBA'] > 10, 'Lipinski_violations'] += 1

df['Lipinski_pass'] = df['Lipinski_violations'] == 0

print('=== Lipinski Rule of Five ===')
print(f'Pass: {df["Lipinski_pass"].sum()}/{len(df)}')
print(f'Fail: {(~df["Lipinski_pass"]).sum()}/{len(df)}')
if (~df['Lipinski_pass']).any():
    print('\nFailing compounds:')
    print(df[~df['Lipinski_pass']][['Name', 'MW', 'cLogP', 'HBD', 'HBA', 'Lipinski_violations']])

print('\n=== Drug-likeness 요약 ===')
for _, row in df.sort_values('Score').head(5).iterrows():
    flag = 'PASS' if row['Lipinski_pass'] else 'FAIL'
    print(f'  {row["Name"]:20s} Score={row["Score"]:6.2f}  LE={row["LE"]:.3f}  Lipinski={flag}')


## 6. 결과 해석 가이드### 6-1. 스코어 해석| 스코어 범위 | 의미 | 조치 ||------------|------|------|| < -10 kcal/mol | 매우 강한 결합 | 실제 약물 수준. 실험 검증 우선 || -8 ~ -10 | 좋은 결합 | 리드 화합물 후보 || -7 ~ -8 | 보통 | 구조 최적화로 개선 가능 || -5 ~ -7 | 약한 결합 | 대폭 수정 필요 || > -5 | 결합 불가 수준 | 다른 scaffold 탐색 |> **주의:** 도킹 스코어는 절대값이 아닌 **상대적 순위(rank-ordering)**로 해석하세요.### 6-2. Ligand Efficiency (LE) 해석| LE 값 | 의미 ||-------|------|| > 0.4 | 우수 (fragment-like 효율) || 0.3 - 0.4 | 좋음 (drug-like) || < 0.3 | 낮음 (분자 크기 대비 결합력 부족) |LE = −Score / HeavyAtoms. 큰 분자가 좋은 스코어를 받는 것은 당연하므로, **LE로 정규화**해야 합니다.### 6-3. R-group 효과 해석 패턴| 관찰 패턴 | 해석 | 예시 ||----------|------|------|| 할로겐(F, Cl) → 스코어 향상 | 전기음성도 + 소수성 증가 | —H → —F: 스코어 0.5~1.0 개선 || —OH, —NH₂ → 스코어 향상 | 추가 수소 결합 형성 | 포켓 내 극성 잔기와 상호작용 || —OCH₃, —CH₃ → 스코어 하락 | 입체 장애(steric clash) | 포켓 입구가 좁은 경우 || R1, R2 동시 최적 → 시너지 | 두 위치가 독립적이지 않음 | 복합 효과 확인 필요 |### 6-4. Activity Cliff**구조는 비슷한데 스코어가 크게 다른 쌍** = Activity Cliff→ 이 쌍이 SAR의 핵심입니다. 작은 구조 변화가 큰 활성 차이를 만드는 이유를 분석하면:- 수소 결합 형성/파괴- 입체 충돌 유발/해소- 전하 상호작용 변화를 알 수 있습니다. **PyMOL에서 두 포즈를 겹쳐서 확인하세요.**### 6-5. Lipinski Rule of Five| 규칙 | 기준 | 위반 시 영향 ||------|------|------------|| MW < 500 | 분자량 | 경구 흡수 불량 || cLogP < 5 | 소수성 | 용해도 문제 || HBD ≤ 5 | H-bond 공여체 | 막 투과성 감소 || HBA ≤ 10 | H-bond 수용체 | 막 투과성 감소 |1개 이하 위반은 허용. 2개 이상 위반 시 경구 투여 약물로 부적합할 가능성 높음.

## 7. 내보내기 (DataWarrior + PyMOL)

In [ ]:
#@title 7-1. SDF 내보내기 (DataWarrior용) {display-mode: "form"}
# 모든 best pose를 하나의 SDF로 합침
combined_sdf = os.path.join(WORK_DIR, 'sar_results.sdf')
writer = Chem.SDWriter(combined_sdf)

for _, row in df.iterrows():
    sdf_file = row['SDF_file']
    if os.path.exists(sdf_file):
        suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
        if len(suppl) > 0 and suppl[0] is not None:
            mol = suppl[0]
            mol.SetProp('Name', row['Name'])
            mol.SetProp('R1', str(row['R1']))
            mol.SetProp('R2', str(row['R2']))
            mol.SetProp('Series', str(row['Series']))
            mol.SetProp('Score', str(row['Score']))
            mol.SetProp('LE', str(row['LE']))
            mol.SetProp('BEI', str(row['BEI']))
            mol.SetProp('MW', str(row['MW']))
            mol.SetProp('cLogP', str(row['cLogP']))
            mol.SetProp('Lipinski_pass', str(row['Lipinski_pass']))
            writer.write(mol)

writer.close()
print(f'Combined SDF: {combined_sdf}')
print(f'  → DataWarrior: File > Open > sar_results.sdf')


### 7-2. CSV 내보내기

7-2. CSV 내보내기


In [ ]:
#@title 7-2. CSV 내보내기 {display-mode: "form"}
csv_path = os.path.join(WORK_DIR, 'sar_table.csv')
export_cols = [c for c in df.columns if c != 'SDF_file']
df[export_cols].to_csv(csv_path, index=False)
print(f'SAR Table CSV: {csv_path}')
print(f'  → DataWarrior: File > Open > sar_table.csv')
print(f'  → Excel/Google Sheets에서도 열 수 있음')


### 7-3. PyMOL 스크립트 생성

7-3. PyMOL 스크립트 생성


In [ ]:
#@title 7-3. PyMOL 스크립트 생성 {display-mode: "form"}
pml_path = os.path.join(WORK_DIR, 'sar_pymol.pml')

# Top 5 compounds
top5 = df.sort_values('Score').head(5)

with open(pml_path, 'w') as f:
    f.write('# SAR Docking Results - PyMOL Script\n')
    f.write('# File > Run Script > sar_pymol.pml\n\n')
    f.write('reinitialize\n')
    f.write('bg_color white\n\n')
    f.write(f'# Protein\n')
    f.write(f'load {prot_H}, protein\n')
    f.write('show cartoon, protein\n')
    f.write('color white, protein\n\n')

    colors = ['green', 'cyan', 'yellow', 'salmon', 'orange']
    f.write('# Top 5 docking poses\n')
    for i, (_, row) in enumerate(top5.iterrows()):
        sdf = row['SDF_file']
        name = row['Name'].replace(' ', '_')
        c = colors[i % len(colors)]
        f.write(f'load {sdf}, {name}\n')
        f.write(f'set all_states, 0\n')
        f.write(f'frame 1\n')
        f.write(f'show sticks, {name}\n')
        f.write(f'color {c}, {name}\n')
        f.write(f'# {row["Name"]}: Score={row["Score"]:.2f}, LE={row["LE"]:.3f}\n\n')

    f.write('# Binding site\n')
    f.write(f'select pocket, protein within 5 of {top5.iloc[0]["Name"].replace(" ","_")}\n')
    f.write('show sticks, pocket\n')
    f.write('color skyblue, pocket\n\n')
    f.write('# H-bonds\n')
    f.write(f'distance hb, {top5.iloc[0]["Name"].replace(" ","_")}, pocket, mode=2\n')
    f.write('set dash_color, yellow, hb\n')
    f.write(f'zoom {top5.iloc[0]["Name"].replace(" ","_")}, 8\n')

print(f'PyMOL script: {pml_path}')
print(f'  → PyMOL: File > Run Script > sar_pymol.pml')
print(f'\nTop 5 compounds visualized:')
for _, row in top5.iterrows():
    print(f'  {row["Name"]:20s} Score={row["Score"]:6.2f}  LE={row["LE"]:.3f}')


### 7-4. 결과 다운로드 (Colab)

7-4. 결과 다운로드 (Colab)


In [ ]:
#@title 7-4. 결과 다운로드 (Colab) {display-mode: "form"}
import shutil

zip_path = '/content/sar_docking_results'
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'Archive: {zip_path}.zip')

try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Not in Colab. Results at: {WORK_DIR}')


## 8. 다음 단계### DataWarrior에서 심화 분석1. `sar_results.sdf`를 DataWarrior에서 열기2. Chemistry → R-Group Decomposition → Core 자동 감지3. Activity Cliff 분석 (Chemistry → Similarity Analysis)4. 추가 속성 계산 (Drug-Likeness, Toxicity Risk)### PyMOL에서 구조 검증1. `sar_pymol.pml`을 PyMOL에서 실행2. Top compound의 수소 결합 패턴 확인3. Activity Cliff 쌍의 포즈 비교4. 결합 부위 표면으로 steric clash 확인### 반복 최적화SAR 분석 결과를 바탕으로:1. 최적 R-group 조합 설계2. 새 SMILES 시리즈 추가3. 이 노트북 재실행4. → 반복하면서 스코어와 LE를 개선